# 05 — Results Analysis

**Objective:** Load all model metrics from `reports/metrics/` and compare models.

- Summary table of all models
- Bar chart comparison across metrics
- Best model selection

In [ ]:
import sys, os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.config import get_paths

paths = get_paths()

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Load All Metrics

In [ ]:
metrics_dir = paths['reports']['metrics_dir']

all_metrics = []
for fname in sorted(os.listdir(metrics_dir)):
    if fname.endswith('_scores.json'):
        with open(os.path.join(metrics_dir, fname), 'r') as f:
            data = json.load(f)
            all_metrics.append(data)
            print(f'Loaded: {fname}')

if not all_metrics:
    print('No metrics files found! Run notebook 04 first.')
else:
    print(f'\nLoaded {len(all_metrics)} model results.')

## 2. Summary Table

In [ ]:
if all_metrics:
    df_results = pd.DataFrame(all_metrics)
    
    # Select key columns
    display_cols = ['model_name', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
    available_cols = [c for c in display_cols if c in df_results.columns]
    df_display = df_results[available_cols].set_index('model_name')
    
    # Format as percentages for readability
    df_pct = df_display.applymap(lambda x: f'{x:.4f}' if isinstance(x, float) else x)
    
    print('=== Model Comparison ===')
    display(df_pct)
    
    # Highlight best
    print('\n--- Best Model per Metric ---')
    for col in df_display.columns:
        best = df_display[col].idxmax()
        print(f'  {col:<20s}: {best} ({df_display.loc[best, col]:.4f})')

## 3. Bar Chart Comparison

In [ ]:
if all_metrics:
    metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
    available_metrics = [m for m in metrics_to_plot if m in df_display.columns]
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    colors = sns.color_palette('viridis', n_colors=len(df_display))
    
    for i, metric in enumerate(available_metrics):
        ax = axes[i]
        values = df_display[metric]
        bars = ax.bar(values.index, values.values, color=colors, edgecolor='black', alpha=0.85)
        ax.set_title(metric.replace('_', ' ').title(), fontsize=13, fontweight='bold')
        ax.set_ylim(0, 1.05)
        ax.set_ylabel('Score')
        
        # Add value labels
        for bar, val in zip(bars, values.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
        
        ax.tick_params(axis='x', rotation=30)
    
    # Hide extra subplot(s) if fewer than 6 metrics
    for j in range(len(available_metrics), len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Model Comparison Across Metrics', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    save_path = os.path.join(paths['reports']['comparisons_dir'], 'model_comparison.png')
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Comparison plot saved to {save_path}')

## 4. Radar Chart (Spider Plot)

In [ ]:
if all_metrics and len(df_display) > 1:
    from matplotlib.patches import FancyBboxPatch
    
    categories = [m.replace('_', ' ').title() for m in available_metrics]
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]  # Close the polygon
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    
    colors_radar = sns.color_palette('Set2', n_colors=len(df_display))
    
    for idx, (model, row) in enumerate(df_display.iterrows()):
        values = row[available_metrics].values.tolist()
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=model, color=colors_radar[idx])
        ax.fill(angles, values, alpha=0.15, color=colors_radar[idx])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_title('Model Performance Radar', fontsize=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    
    plt.tight_layout()
    save_path = os.path.join(paths['reports']['comparisons_dir'], 'model_radar.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Radar plot saved to {save_path}')